### Infinity Throughput Sweep Benchmark Notebook

这个 notebook 参照 `inference_throughput_Infinity.ipynb`，用于递增 batch size 吞吐测试：
- 从 `bs=1` 开始递增
- 仅在 `bs=1` 时 warmup 一次
- 每个 `bs` 正式测试 2 次，取平均时间与 IPS
- 遇到首次 OOM 即停止
- 记录并输出每个 `bs` 的 throughput 结果


In [ ]:
import os
import sys
import time
import argparse

import numpy as np
import torch
import os.path as osp

# 通过环境变量选择 GPU，例如: GPU_ID=1
GPU_ID = int(os.environ.get("GPU_ID", "3"))
torch.cuda.set_device(GPU_ID)

project_root = '/home/jiaji_lu/AR/VAR-Q'
os.chdir(project_root)
sys.path.append(project_root)

from Infinity.tools.run_infinity import *
from Infinity.tools.run_infinity import _import_dynamic_resolution

############# Configuration File #############
CONFIG_FILE = "/home/jiaji_lu/AR/VAR-Q/temp/Infinity-VAR_Q-8.json"

print(f"[Config] Loading configuration from {CONFIG_FILE}")

var_q_path = '/home/jiaji_lu/AR/VAR-Q/VAR_Q'
if os.path.exists(var_q_path):
    sys.path.append(var_q_path)
    from config_loader import VARQConfig

    try:
        config = VARQConfig(CONFIG_FILE)

        model_config = config.get_model_config()
        quant_config = config.get_quantization_config()
        inference_config = config.get_inference_config()
        batch_config = config.get_batch_processing_config()
        checkpoint_config = config.get_checkpoint_config()

        print("[Config] Configuration loaded successfully!")
        print(f"[Config] Model: {model_config.get('model_type')}")
        print(f"[Config] VAR-Q Quantization: {'enabled' if quant_config.get('enable') else 'disabled'}")
        print(f"[Config] Fused int4 KV FlashAttention: {'enabled' if quant_config.get('enable_fused_kv_flashattn', False) else 'disabled'}")

    except Exception as e:
        print(f"[Error] Failed to load configuration: {e}")
        raise
else:
    print(f"[Error] VAR_Q path not found: {var_q_path}")
    raise FileNotFoundError(f"VAR_Q directory not found at {var_q_path}")


In [ ]:
############# Create args from configuration #############
model_type = model_config.get('model_type', 'infinity_2b')

if model_type == "infinity_2b":
    vae_type = 32
    apply_spatial_patchify = 0
    checkpoint_type = "torch"
elif model_type == "infinity_8b":
    vae_type = 14
    apply_spatial_patchify = 1
    checkpoint_type = "torch_shard"
else:
    vae_type = 32
    apply_spatial_patchify = 0
    checkpoint_type = "torch"

args = argparse.Namespace(
    model_type=model_type,
    pn='1M',
    model_path=checkpoint_config.get('model_path'),
    vae_path=checkpoint_config.get('vae_ckpt'),
    text_encoder_ckpt='/data/boxunxu/Infinity/flan-t5-xl',

    vae_type=vae_type,
    apply_spatial_patchify=apply_spatial_patchify,
    checkpoint_type=checkpoint_type,

    add_lvl_embeding_only_first_block=1,
    use_bit_label=1,
    rope2d_each_sa_layer=1,
    rope2d_normalized_by_hw=2,
    use_scale_schedule_embedding=0,
    sampling_per_bits=1,
    text_channels=2048,
    h_div_w_template=inference_config.get('h_div_w', 1.0),
    use_flex_attn=0,

    cache_dir='/dev/shm',
    seed=inference_config.get('seed', 0),
    bf16=1,
    save_file='tmp.jpg',
    enable_model_cache=0,

    cfg_insertion_layer=0,
    enable_positive_prompt=0,
    cfg=inference_config.get('cfg', 3.0),
    tau=inference_config.get('tau', 0.5),

    enable_quantization=int(quant_config.get('enable', False)),
    q_bits=quant_config.get('q_bits', 8),
    quant_method=quant_config.get('quant_method', 'G_SCALE_HEAD_DIM'),
    qkv_format=quant_config.get('qkv_format', 'BLHc'),
    rescale_qk=int(quant_config.get('rescale_qk', False)),
    enable_fused_kv_flashattn=int(quant_config.get('enable_fused_kv_flashattn', False)),
)

print("[Args] Arguments created from configuration:")
print(f"  - Model: {args.model_type}")
print(f"  - Model path: {args.model_path}")
print(f"  - VAE path: {args.vae_path}")
print(f"  - VAE type: {args.vae_type}")
print(f"  - Seed: {args.seed}")
print(f"  - q_bits: {args.q_bits}")
print(f"  - enable_fused_kv_flashattn: {args.enable_fused_kv_flashattn}")

if args.enable_fused_kv_flashattn and int(args.q_bits) != 4:
    raise ValueError("enable_fused_kv_flashattn requires q_bits == 4")


In [ ]:
############# load model #############
print("[Loading tokenizer and text encoder]")
text_tokenizer, text_encoder = load_tokenizer(t5_path=args.text_encoder_ckpt)

print("[Loading VAE]")
vae = load_visual_tokenizer(args)

print("[Loading Infinity]")
infinity = load_transformer(vae, args)

# 预加载动态分辨率模板，避免把一次性初始化记入吞吐统计
if 'dynamic_resolution_h_w' not in globals() or dynamic_resolution_h_w is None:
    dynamic_resolution_h_w, h_div_w_templates = _import_dynamic_resolution()

print("[Model] All modules loaded. Ready for throughput sweep benchmark.")


In [ ]:
############# FlashAttention 路径与耗时监测 #############
import flash_attn
import flash_attn.flash_attn_interface as _fai
import Infinity.infinity.models.basic as _inf_basic

FA_MONITOR = {
    "baseline": {"calls": 0, "total_ms": 0.0},
    "fused_int4": {"calls": 0, "total_ms": 0.0},
    "fused_single_group": {"calls": 0, "total_ms": 0.0},
    "fused_token_group": {"calls": 0, "total_ms": 0.0},
    "group_block_stats": {
        "total_blocks": 0,
        "single_group_blocks": 0,
        "two_group_blocks": 0,
        "sum_groups_per_block": 0,
        "max_groups_per_block": 0,
    },
    "estimated_kernel_paths": {
        "single": 0,
        "segmented": 0,
        "general": 0,
    },
    "originals": {},
}

# 吞吐测试必须关：每个 FA 后 Event.synchronize + 下面 block 统计里的 torch.unique/.item 会强制 CPU-GPU 同步，
# 在整图自回归（百万级 FA 调用）下会把墙钟时间放大到数百秒，与 kernel 本身无关。
# 默认关闭，需要调试时通过 install_fa_monitor(..., per_call_timing=True, collect_block_stats=True) 开启。
_FA_PER_CALL_TIMING = False
_FA_COLLECT_BLOCK_STATS = False


def _is_token_group_call(kwargs):
    kv_group = kwargs.get("kv_group", None)
    return kv_group is not None


def _estimate_block_n(q, kwargs):
    causal = bool(kwargs.get("causal", False))
    # Fused inference path enforces dropout=0.
    return int(_fai._get_block_size_n(q.device, int(q.shape[-1]), False, causal))


def _update_group_block_stats(q, k_packed, kwargs):
    kv_group = kwargs.get("kv_group", None)
    bsz = int(k_packed.shape[0])
    seqlen_k = int(k_packed.shape[1])
    block_n = max(1, _estimate_block_n(q, kwargs))

    total_blocks = 0
    single_blocks = 0
    two_blocks = 0
    sum_groups = 0
    max_groups = 0

    est_single = 0
    est_segmented = 0
    est_general = 0

    segmented_threshold = 4

    if kv_group is None:
        blocks_per_seq = (seqlen_k + block_n - 1) // block_n
        total_blocks = bsz * blocks_per_seq
        single_blocks = total_blocks
        sum_groups = total_blocks
        max_groups = 1 if total_blocks > 0 else 0
        est_single = total_blocks
    else:
        kg = kv_group
        if kg.dim() != 2:
            return
        for b in range(int(kg.shape[0])):
            for st in range(0, seqlen_k, block_n):
                ed = min(st + block_n, seqlen_k)
                blk = kg[b, st:ed]
                g_cnt = int(torch.unique(blk).numel())
                seg_cnt = 1 if int(blk.numel()) > 0 else 0
                if int(blk.numel()) > 1:
                    seg_cnt += int((blk[1:] != blk[:-1]).sum().item())

                total_blocks += 1
                sum_groups += g_cnt
                if g_cnt == 1:
                    single_blocks += 1
                if g_cnt == 2:
                    two_blocks += 1
                if g_cnt > max_groups:
                    max_groups = g_cnt

                if g_cnt == 1:
                    est_single += 1
                elif seg_cnt <= segmented_threshold and g_cnt <= segmented_threshold:
                    est_segmented += 1
                else:
                    est_general += 1

    s = FA_MONITOR["group_block_stats"]
    s["total_blocks"] += total_blocks
    s["single_group_blocks"] += single_blocks
    s["two_group_blocks"] += two_blocks
    s["sum_groups_per_block"] += sum_groups
    s["max_groups_per_block"] = max(s["max_groups_per_block"], max_groups)

    p = FA_MONITOR["estimated_kernel_paths"]
    p["single"] += est_single
    p["segmented"] += est_segmented
    p["general"] += est_general


def _wrap_fa(tag, fn):
    if getattr(fn, "_fa_wrapped", False):
        return fn

    def _wrapped(*args, **kwargs):
        elapsed = 0.0
        if torch.cuda.is_available() and _FA_PER_CALL_TIMING:
            st = torch.cuda.Event(enable_timing=True)
            ed = torch.cuda.Event(enable_timing=True)
            st.record()
            out = fn(*args, **kwargs)
            ed.record()
            ed.synchronize()
            elapsed = float(st.elapsed_time(ed))
            FA_MONITOR[tag]["total_ms"] += elapsed
        else:
            out = fn(*args, **kwargs)
        FA_MONITOR[tag]["calls"] += 1

        if tag == "fused_int4":
            sub_tag = "fused_token_group" if _is_token_group_call(kwargs) else "fused_single_group"
            FA_MONITOR[sub_tag]["calls"] += 1
            FA_MONITOR[sub_tag]["total_ms"] += elapsed
            if _FA_COLLECT_BLOCK_STATS and len(args) >= 2:
                _update_group_block_stats(args[0], args[1], kwargs)

        return out

    _wrapped._fa_wrapped = True
    return _wrapped


def install_fa_monitor(reset_stats=True, run_fused_flag=None,
                       per_call_timing=False, collect_block_stats=False):
    global _FA_PER_CALL_TIMING, _FA_COLLECT_BLOCK_STATS
    _FA_PER_CALL_TIMING = per_call_timing
    _FA_COLLECT_BLOCK_STATS = collect_block_stats

    if "flash_attn_func" not in FA_MONITOR["originals"]:
        FA_MONITOR["originals"]["flash_attn_func"] = flash_attn.flash_attn_func
    if "flash_attn_func_quant_kv_int4" not in FA_MONITOR["originals"]:
        FA_MONITOR["originals"]["flash_attn_func_quant_kv_int4"] = flash_attn.flash_attn_func_quant_kv_int4

    baseline = _wrap_fa("baseline", FA_MONITOR["originals"]["flash_attn_func"])
    fused = _wrap_fa("fused_int4", FA_MONITOR["originals"]["flash_attn_func_quant_kv_int4"])

    # Patch 到全部可能调用入口
    flash_attn.flash_attn_func = baseline
    flash_attn.flash_attn_func_quant_kv_int4 = fused
    _fai.flash_attn_func = baseline
    _fai.flash_attn_func_quant_kv_int4 = fused
    _inf_basic.flash_attn_func = baseline

    if reset_stats:
        FA_MONITOR["baseline"] = {"calls": 0, "total_ms": 0.0}
        FA_MONITOR["fused_int4"] = {"calls": 0, "total_ms": 0.0}
        FA_MONITOR["fused_single_group"] = {"calls": 0, "total_ms": 0.0}
        FA_MONITOR["fused_token_group"] = {"calls": 0, "total_ms": 0.0}
        FA_MONITOR["group_block_stats"] = {
            "total_blocks": 0,
            "single_group_blocks": 0,
            "two_group_blocks": 0,
            "sum_groups_per_block": 0,
            "max_groups_per_block": 0,
        }
        FA_MONITOR["estimated_kernel_paths"] = {
            "single": 0,
            "segmented": 0,
            "general": 0,
        }

    print("[FA Monitor] installed")
    effective_flag = run_fused_flag if run_fused_flag is not None else getattr(args, 'enable_fused_kv_flashattn', 'N/A')
    print(f"[FA Monitor] enable_fused_kv_flashattn={effective_flag}")
    print(f"[FA Monitor] per_call_timing={_FA_PER_CALL_TIMING}, collect_block_stats={_FA_COLLECT_BLOCK_STATS}")


def _fmt_line(tag):
    item = FA_MONITOR[tag]
    avg = item["total_ms"] / item["calls"] if item["calls"] > 0 else 0.0
    return f"  - {tag}: calls={item['calls']}, total_ms={item['total_ms']:.3f}, avg_ms={avg:.3f}"


def print_fa_monitor_summary(run_fused_flag=None):
    print("\n[FA Monitor] Summary")
    print(_fmt_line("baseline"))
    print(_fmt_line("fused_int4"))
    print(_fmt_line("fused_single_group"))
    print(_fmt_line("fused_token_group"))

    s = FA_MONITOR["group_block_stats"]
    tb = max(1, int(s["total_blocks"]))
    single_group_block_ratio = s["single_group_blocks"] / tb
    two_group_block_ratio = s["two_group_blocks"] / tb
    avg_groups_per_block = s["sum_groups_per_block"] / tb
    max_groups_per_block = s["max_groups_per_block"]
    print("[FA Monitor][GroupBlockStats]")
    print(f"  - total_blocks={s['total_blocks']}")
    print(f"  - single_group_block_ratio={single_group_block_ratio:.6f}")
    print(f"  - two_group_block_ratio={two_group_block_ratio:.6f}")
    print(f"  - avg_groups_per_block={avg_groups_per_block:.6f}")
    print(f"  - max_groups_per_block={max_groups_per_block}")

    p = FA_MONITOR["estimated_kernel_paths"]
    pt = max(1, int(p["single"] + p["segmented"] + p["general"]))
    print("[FA Monitor][EstimatedKernelPaths]")
    print(f"  - single={p['single']} ({p['single']/pt:.6f})")
    print(f"  - segmented={p['segmented']} ({p['segmented']/pt:.6f})")
    print(f"  - general={p['general']} ({p['general']/pt:.6f})")

    effective_flag = run_fused_flag if run_fused_flag is not None else getattr(args, 'enable_fused_kv_flashattn', 0)
    if int(effective_flag) == 1 and FA_MONITOR["fused_int4"]["calls"] == 0:
        print("[FA Monitor][WARN] fused 开关已开启，但 fused_int4 调用为 0（没有命中 fused kernel）。")


install_fa_monitor(reset_stats=True, per_call_timing=False, collect_block_stats=False)


In [ ]:
############# A/B 对比：baseline vs fused（不改本地 config 文件）#############
import copy
import gc


def _gpu_mem_gib():
    if not torch.cuda.is_available():
        return 0.0, 0.0
    dev = torch.cuda.current_device()
    allocated = torch.cuda.memory_allocated(dev) / (1024 ** 3)
    reserved = torch.cuda.memory_reserved(dev) / (1024 ** 3)
    return allocated, reserved


def _cleanup_cuda_runtime(tag=""):
    # 释放 notebook 中常驻对象，防止 A/B 叠模型
    for name in ["infinity", "ab_baseline", "ab_fused", "_warm", "img_batch"]:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()
        a, r = _gpu_mem_gib()
        print(f"[AB][cleanup]{' ' + tag if tag else ''} allocated={a:.2f} GiB, reserved={r:.2f} GiB")


def _clone_args_with_fused(enable_fused: bool):
    if "args" not in globals():
        raise RuntimeError("missing required global: args（请先执行配置/模型初始化单元）")
    local = argparse.Namespace(**vars(args))
    local.enable_fused_kv_flashattn = int(bool(enable_fused))
    return local


@torch.no_grad()
def _encode_prompt_batch_with_encoder(prompts, text_tokenizer_, text_encoder_, use_positive_prompt=False):
    if use_positive_prompt:
        prompts = [aug_with_positive_prompt(p) for p in prompts]

    tokens = text_tokenizer_(
        text=prompts,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = tokens.input_ids.cuda(non_blocking=True)
    mask = tokens.attention_mask.cuda(non_blocking=True)

    text_features = text_encoder_(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    Ltext = max(lens)

    kv_compact = []
    for len_i, feat_i in zip(lens, text_features.unbind(0)):
        kv_compact.append(feat_i[:len_i])
    kv_compact = torch.cat(kv_compact, dim=0)

    return kv_compact, lens, cu_seqlens_k, Ltext


def run_sweep_once_with_fused_flag(
    enable_fused: bool,
    bs_list=(1, 3, 6),
    runs_per_bs: int = 2,
    warmup_bs1_runs: int = 1,
):
    local_args = _clone_args_with_fused(enable_fused)
    run_tag = "fused" if enable_fused else "baseline"
    bs_list = sorted(set(int(x) for x in bs_list))

    # 一次性依赖检查（避免 NameError）
    missing = [
        name for name in ["vae", "text_tokenizer", "text_encoder"]
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            "missing required globals for AB sweep: " + ", ".join(missing) +
            "（请先执行模型/输入准备单元）"
        )

    # 兼容默认值：避免反复因为 notebook 执行顺序导致中断
    base_seed = int(globals().get("seed", getattr(local_args, "seed", 0)))
    base_prompt = str(globals().get("prompt", "A cinematic portrait, ultra detailed, studio lighting"))
    use_positive_prompt_flag = bool(globals().get("enable_positive_prompt", 0))
    cfg_src = globals().get("cfg_list", globals().get("cfg", 3.0))
    tau_src = globals().get("tau_list", globals().get("tau", 1.0))

    if "scale_schedule" in globals() and globals()["scale_schedule"] is not None:
        scale_schedule_local = globals()["scale_schedule"]
    else:
        if "dynamic_resolution_h_w" not in globals() or globals()["dynamic_resolution_h_w"] is None:
            globals()["dynamic_resolution_h_w"], globals()["h_div_w_templates"] = _import_dynamic_resolution()
        dr = globals()["dynamic_resolution_h_w"]
        h_key = getattr(local_args, "h_div_w_template", 1.0)
        if h_key not in dr and str(h_key) in dr:
            h_key = str(h_key)
        if h_key not in dr:
            try:
                h_val = float(h_key)
                num_keys = [k for k in dr.keys() if isinstance(k, (int, float))]
                if len(num_keys) > 0:
                    h_key = min(num_keys, key=lambda x: abs(float(x) - h_val))
                else:
                    h_key = list(dr.keys())[0]
            except Exception:
                h_key = list(dr.keys())[0]
        pn_key = getattr(local_args, "pn", "1M")
        if pn_key not in dr[h_key]:
            pn_key = list(dr[h_key].keys())[0]
        raw_scales = dr[h_key][pn_key]["scales"]
        scale_schedule_local = [(1, h, w) for (_, h, w) in raw_scales]
        globals()["scale_schedule"] = scale_schedule_local

    sched_len = max(1, len(scale_schedule_local))

    def _to_list(x, default, length):
        if x is None:
            return [default] * length
        if isinstance(x, (int, float)):
            return [float(x)] * length
        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return [default] * length
            if len(x) >= length:
                return list(x)[:length]
            return list(x) + [float(x[-1])] * (length - len(x))
        return [default] * length

    cfg_list_local = _to_list(cfg_src, getattr(local_args, "cfg", 3.0), sched_len)
    tau_list_local = _to_list(tau_src, getattr(local_args, "tau", 1.0), sched_len)

    print(f"\n[AB] ===== Run {run_tag} (enable_fused_kv_flashattn={local_args.enable_fused_kv_flashattn}) =====")
    print(f"[AB][{run_tag}] target bs list: {bs_list}, base_seed={base_seed}")

    # 每次 run 前先清理，避免和上一次 run 叠显存
    _cleanup_cuda_runtime(tag=f"before_{run_tag}")

    # 只重载 transformer，避免改本地配置文件
    local_infinity = load_transformer(vae, local_args)

    # 安装/重置 FA 监测（使用本轮 local_args，避免日志与实际开关不一致）
    # 吞吐 sweep 关闭 per_call_timing 和 collect_block_stats 避免 GPU sync 开销
    install_fa_monitor(reset_stats=True, run_fused_flag=local_args.enable_fused_kv_flashattn,
                       per_call_timing=False, collect_block_stats=False)

    local_results = []
    local_oom_bs = None

    @torch.no_grad()
    def _run_one_batch(cur_seed: int, current_bs: int):
        prompts = [base_prompt] * current_bs
        text_cond_tuple = _encode_prompt_batch_with_encoder(
            prompts,
            text_tokenizer,
            text_encoder,
            use_positive_prompt=use_positive_prompt_flag,
        )

        with torch.amp.autocast("cuda", enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            _, _, img_batch = local_infinity.autoregressive_infer_cfg(
                vae=vae,
                scale_schedule=scale_schedule_local,
                label_B_or_BLT=text_cond_tuple,
                g_seed=cur_seed,
                B=current_bs,
                negative_label_B_or_BLT=None,
                force_gt_Bhw=None,
                cfg_sc=3,
                cfg_list=cfg_list_local,
                tau_list=tau_list_local,
                top_k=900,
                top_p=0.97,
                returns_vemb=1,
                ratio_Bl1=None,
                gumbel=0,
                norm_cfg=False,
                cfg_exp_k=0.0,
                cfg_insertion_layer=[local_args.cfg_insertion_layer],
                vae_type=local_args.vae_type,
                softmax_merge_topk=-1,
                ret_img=True,
                trunk_scale=1000,
                gt_leak=0,
                gt_ls_Bl=None,
                inference_mode=True,
                sampling_per_bits=local_args.sampling_per_bits,
            )
        return img_batch

    for bs in bs_list:
        print(f"[AB][{run_tag}] bs={bs}")
        try:
            if bs == 1 and warmup_bs1_runs > 0:
                for w in range(warmup_bs1_runs):
                    _warm = _run_one_batch(base_seed + w, bs)
                    torch.cuda.synchronize()
                    del _warm
                torch.cuda.empty_cache()

            run_times = []
            peak_alloc_bytes_list = []
            for r in range(runs_per_bs):
                cur_seed = base_seed + 10000 * bs + r
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()

                t0 = time.perf_counter()
                img_batch = _run_one_batch(cur_seed, bs)
                torch.cuda.synchronize()
                t1 = time.perf_counter()

                run_times.append(t1 - t0)
                peak_alloc_bytes_list.append(torch.cuda.max_memory_allocated())
                del img_batch

            avg_time_per_batch = float(np.mean(run_times))
            ips = bs / avg_time_per_batch if avg_time_per_batch > 0 else float('inf')
            peak_alloc_mib = max(peak_alloc_bytes_list) / (1024 ** 2)

            local_results.append({
                "bs": bs,
                "avg_batch_time_s": avg_time_per_batch,
                "ips": ips,
                "peak_alloc_mib": peak_alloc_mib,
            })
            print(f"    -> {run_tag}: avg={avg_time_per_batch:.4f}s, IPS={ips:.4f}, peak={peak_alloc_mib:.1f} MiB")

        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            local_oom_bs = bs
            print(f"    -> {run_tag}: OOM at bs={bs}, stop.")
            break
        except Exception as e:
            import traceback
            print(f"    -> {run_tag}: ERROR at bs={bs}: {type(e).__name__}: {e}")
            traceback.print_exc()
            raise

    monitor_snapshot = copy.deepcopy(FA_MONITOR)

    # 清理模型，避免 A/B 互相污染显存
    del local_infinity
    _cleanup_cuda_runtime(tag=f"after_{run_tag}")

    return {
        "tag": run_tag,
        "args": local_args,
        "results": local_results,
        "oom_bs": local_oom_bs,
        "fa_monitor": monitor_snapshot,
        "runs_per_bs": runs_per_bs,
    }


In [ ]:
############# 执行 A/B 对比并输出结果（容错版）#############
# 固定对比 batch size
ab_bs_list = [1, 3, 6]
ab_runs_per_bs = 1
ab_warmup_bs1_runs = 1

ab_baseline = None
ab_fused = None

try:
    ab_baseline = run_sweep_once_with_fused_flag(
        enable_fused=False,
        bs_list=ab_bs_list,
        runs_per_bs=ab_runs_per_bs,
        warmup_bs1_runs=ab_warmup_bs1_runs,
    )
except Exception as e:
    print(f"[AB][baseline][ERROR] {type(e).__name__}: {e}")

try:
    ab_fused = run_sweep_once_with_fused_flag(
        enable_fused=True,
        bs_list=ab_bs_list,
        runs_per_bs=ab_runs_per_bs,
        warmup_bs1_runs=ab_warmup_bs1_runs,
    )
except Exception as e:
    print(f"[AB][fused][ERROR] {type(e).__name__}: {e}")

ab_baseline_obj = globals().get("ab_baseline", None)
ab_fused_obj = globals().get("ab_fused", None)

packs = [p for p in [ab_baseline_obj, ab_fused_obj] if p is not None]
if len(packs) == 0:
    raise RuntimeError("[AB] baseline/fused 都未成功执行，请查看上面的 ERROR 日志。")

print("\n[AB] ===== FA Monitor Compare =====")
for pack in packs:
    mon = pack["fa_monitor"]
    b = mon["baseline"]
    f = mon["fused_int4"]
    fs = mon.get("fused_single_group", {"calls": 0, "total_ms": 0.0})
    ft = mon.get("fused_token_group", {"calls": 0, "total_ms": 0.0})
    s = mon.get("group_block_stats", {
        "total_blocks": 0,
        "single_group_blocks": 0,
        "two_group_blocks": 0,
        "sum_groups_per_block": 0,
        "max_groups_per_block": 0,
    })
    p = mon.get("estimated_kernel_paths", {
        "single": 0,
        "segmented": 0,
        "general": 0,
    })
    tb = max(1, int(s["total_blocks"]))
    pt = max(1, int(p["single"] + p["segmented"] + p["general"]))
    single_group_block_ratio = s["single_group_blocks"] / tb
    two_group_block_ratio = s["two_group_blocks"] / tb
    avg_groups_per_block = s["sum_groups_per_block"] / tb
    max_groups_per_block = s["max_groups_per_block"]
    print(
        f"[AB][{pack['tag']}] baseline_calls={b['calls']}, fused_calls={f['calls']}, "
        f"fused_single_group_calls={fs['calls']}, fused_token_group_calls={ft['calls']}, "
        f"baseline_ms={b['total_ms']:.3f}, fused_ms={f['total_ms']:.3f}, "
        f"single_group_block_ratio={single_group_block_ratio:.6f}, two_group_block_ratio={two_group_block_ratio:.6f}, "
        f"avg_groups_per_block={avg_groups_per_block:.6f}, max_groups_per_block={max_groups_per_block}, "
        f"est_single_ratio={p['single']/pt:.6f}, est_segmented_ratio={p['segmented']/pt:.6f}, est_general_ratio={p['general']/pt:.6f}"
    )

base_map = {x['bs']: x for x in (ab_baseline_obj['results'] if ab_baseline_obj is not None else [])}
fuse_map = {x['bs']: x for x in (ab_fused_obj['results'] if ab_fused_obj is not None else [])}
common_bs = sorted(set(base_map.keys()) & set(fuse_map.keys()))

if len(common_bs) == 0:
    print("[AB][WARN] baseline/fused 没有可对齐的 bs 结果（某一侧可能失败或 OOM）。")
else:
    print("\n[AB] ===== Throughput Compare (same bs) =====")
    print("bs | baseline_ips | fused_ips | speedup(fused/baseline)")
    for bs in common_bs:
        b_ips = base_map[bs]['ips']
        f_ips = fuse_map[bs]['ips']
        speedup = (f_ips / b_ips) if b_ips > 0 else float('inf')
        print(f"{bs:2d} | {b_ips:11.4f} | {f_ips:9.4f} | {speedup:8.4f}x")

# 只有在有对齐结果时才写 CSV
if len(common_bs) > 0:
    ab_out_dir = "/home/jiaji_lu/AR/VAR-Q/Benchmark/results_txt/throughput"
    os.makedirs(ab_out_dir, exist_ok=True)
    ab_csv = osp.join(ab_out_dir, f"infinity-throughput-ab-{args.quant_method}-{args.q_bits}b.csv")
    with open(ab_csv, "w", encoding="utf-8") as f:
        f.write("bs,baseline_ips,fused_ips,speedup_fused_over_baseline\n")
        for bs in common_bs:
            b_ips = base_map[bs]['ips']
            f_ips = fuse_map[bs]['ips']
            speedup = (f_ips / b_ips) if b_ips > 0 else float('inf')
            f.write(f"{bs},{b_ips:.6f},{f_ips:.6f},{speedup:.6f}\n")
    print(f"\n[AB] Saved compare csv: {ab_csv}")
